In [12]:
import warnings
warnings.filterwarnings("ignore", category=Warning)
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
import torch
import torch.nn as nn

In [14]:
#First step is to load the data into an oject and setup our X and y (not used in our learning)
wineData = load_wine()
#print(wineData.DESCR)

X = pd.DataFrame(wineData.data, columns=wineData.feature_names)
y = pd.Series(wineData.target)
X.head()

#Using the shape command, you can investigate the dimensionality
print('Shape of Data (X):',wineData.data.shape)
#print('Shape of Target Data (y)', wineData.target.shape)

Shape of Data (X): (178, 13)


In [15]:
X.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [16]:
y.value_counts()

1    71
0    59
2    48
Name: count, dtype: int64

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [19]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

In [30]:
class SimpleANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(13, 16)
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 3)

    def forward(self, x):
        x = self.layer1(x)
        x = torch.relu(x)

        x = self.layer2(x)
        x = torch.relu(x)

        logits = self.output(x)

        return logits

In [31]:
model = SimpleANN()

print(model)

SimpleANN(
  (layer1): Linear(in_features=13, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=3, bias=True)
)


In [32]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [33]:
epochs = 200

for epoch in range(epochs):

    # Forward pass
    outputs = model(X_train)

    # Calculate loss
    loss = loss_fn(outputs, y_train)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Loss: {loss.item():.4f}"
        )

Epoch [20/200], Loss: 0.2379
Epoch [40/200], Loss: 0.0124
Epoch [60/200], Loss: 0.0025
Epoch [80/200], Loss: 0.0013
Epoch [100/200], Loss: 0.0009
Epoch [120/200], Loss: 0.0007
Epoch [140/200], Loss: 0.0006
Epoch [160/200], Loss: 0.0005
Epoch [180/200], Loss: 0.0004
Epoch [200/200], Loss: 0.0003


In [34]:
with torch.no_grad():

    outputs = model(X_test)

    predictions = torch.argmax(outputs, dim=1)

In [35]:
print(predictions)

tensor([0, 1, 0, 1, 1, 0, 0, 1, 1, 2, 1, 2, 0, 2, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
        1, 1, 0, 2, 1, 2, 0, 2, 1, 2, 2, 2])


In [36]:
print(y_test)

tensor([0, 2, 0, 1, 1, 0, 0, 1, 1, 2, 1, 2, 0, 2, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0,
        1, 1, 0, 2, 1, 2, 0, 2, 1, 2, 2, 2])
